# Quickstart: running each method directly

This notebook shows the minimal call for each of the six methods in this
repo, on a small toy graph population -- without the sweep, checkpointing,
or joblib parallelization machinery used in `simulations/`. It exists so a
new user (or reviewer) can see the actual input/output shape each method
expects without reverse-engineering it from a sweep script.

**Run this from the repository root** (`jupyter notebook` launched from
`msc-thesis-code/`), since the imports below are package-relative -- see
the README's "Running the code" section for why.

The six methods split into two families with different call shapes:

- **Full-sample hypothesis tests** (`ginestet2017`, `dubey2019`,
  `lovato2020`): take the whole graph population `(G_all, y)` and return a
  p-value directly. No train/test split -- these are hypothesis tests, not
  classifiers.
- **Classifier two-sample tests** (`knn`, `kernel_svm`, `gcn`): take a
  precomputed distance/kernel matrix (or the raw graphs, for GCN) plus a
  fixed train/test split, return predictions, and need to be passed through
  `permutation_test` separately to get a p-value.


## Setup: generate a small toy graph population

In [1]:
import sys
sys.path.insert(0, '.')  # repo root -- redundant if launched from root, harmless if not

import numpy as np
import networkx as nx

from data_generation.barabasi_albert_datagen import generate_data

# Two populations of Barabasi-Albert graphs at different power-law exponents
# (gamma1=2.5 vs gamma2=2.0 -- a clear separation, so results are non-trivial
# but this still runs in seconds). Small n_samples and n_nodes keep every
# cell below fast.
data = generate_data(n_samples=25, n_nodes=10, test_size=0.5,
                      random_state=0, gamma1=2.5, gamma2=2.0, m=2)

G_all = data["G_all"]      # list of networkx graphs, both groups pooled
y = data["y"]               # 0/1 group labels, same order as G_all
idx_train = data["idx_train"]
idx_test = data["idx_test"]

print(f"{len(G_all)} graphs total, {sum(y==0)} vs {sum(y==1)}, "
      f"{len(idx_train)} train / {len(idx_test)} test")


50 graphs total, 25 vs 25, 25 train / 25 test


## Shared precomputation

`ginestet2017`, `dubey2019`, and `lovato2020` all operate on graph
Laplacians; `knn` and `kernel_svm` need a distance/kernel matrix built from
those same Laplacians. Computing this once and reusing it (rather than
recomputing per method) mirrors what the sweep scripts do internally.

In [2]:
laplacians = [nx.laplacian_matrix(G).toarray() for G in G_all]
n = len(laplacians)

D_sq = np.zeros((n, n))
for i in range(n):
    for j in range(i + 1, n):
        d_sq = np.sum((laplacians[i] - laplacians[j]) ** 2)
        D_sq[i, j] = D_sq[j, i] = d_sq

D_matrix = np.sqrt(D_sq)

nonzero = D_sq[D_sq > 0]
gamma = 1.0 / np.median(nonzero)
K_matrix = np.exp(-gamma * D_sq)

print("Laplacians, distance matrix, and kernel matrix ready.")


Laplacians, distance matrix, and kernel matrix ready.


## 1. Ginestet et al. (2017) -- Frechet-mean Wald test

In [3]:
from methods import ginestet2017

result = ginestet2017.run_test(G_all, y, laplacians=laplacians)
print(f"T2 statistic = {result['statistic']:.3f}, p-value = {result['p_value']:.4f}, dof = {result['dof']}")


T2 statistic = 375.919, p-value = 0.0000, dof = 45


## 2. Dubey & Muller (2019) -- Frechet ANOVA

**Note on sample size:** this test's variance estimator can become
numerically degenerate at very small within-group sample sizes (it raises a
clear `ValueError` naming the affected group, rather than silently
returning a meaningless p-value -- see `methods/dubey2019.py` for details).
This was not observed at any sample size actually used in the thesis
(checked at n=20/25/50/100/450); it is mentioned here only because it can
occur at the very small toy sample size used for speed in this notebook.

In [4]:
from methods import dubey2019

result = dubey2019.run_test(G_all, y, laplacians=laplacians)
print(f"p-value = {result['p_value']:.4f}")


p-value = 0.0000


## 3. Lovato et al. (2020) -- IP-Student / IP-Fisher permutation test

Unlike the two tests above, this one returns `p_combined` (the Tippett
combination of the mean- and variance-targeting statistics), not `p_value`.

In [5]:
from methods import lovato2020

result = lovato2020.run_test(G_all, y, B=200, random_state=0, laplacians=laplacians)
print(f"combined p-value = {result['p_combined']:.4f}")


combined p-value = 0.0050


## 4. K-Nearest Neighbors (WL/Frobenius distance)

Classifier-style methods fit once on the training split, predict once on
the test split, then need `permutation_test` to turn the observed
misclassification error into a p-value.

In [6]:
from methods import knn
from testing.permutation_test import permutation_test

y_pred, y_test = knn.get_predictions(D_matrix, y, idx_train, idx_test)
observed_error, p_value, _ = permutation_test(y_pred, y_test, B=200, random_state=0)
print(f"test error = {observed_error:.3f}, p-value = {p_value:.4f}")


test error = 0.240, p-value = 0.0100


## 5. Kernel SVM (precomputed WL kernel)

In [7]:
from methods import kernel_svm

y_pred, y_test = kernel_svm.get_predictions(K_matrix, y, idx_train, idx_test)
observed_error, p_value, _ = permutation_test(y_pred, y_test, B=200, random_state=0)
print(f"test error = {observed_error:.3f}, p-value = {p_value:.4f}")


test error = 0.160, p-value = 0.0050


## 6. Graph Convolutional Network

GCN operates on the raw graphs directly (not a precomputed distance/kernel
matrix), and trains a small model internally. `epochs` is set low here
purely so this cell finishes quickly -- see `simulations/` for the epoch
counts actually used in the thesis results.

In [8]:
from methods import gcn

y_pred, y_test = gcn.get_predictions(G_all, y, idx_train, idx_test, epochs=20, seed=0)
observed_error, p_value, _ = permutation_test(y_pred, y_test, B=200, random_state=0)
print(f"test error = {observed_error:.3f}, p-value = {p_value:.4f}")


test error = 0.000, p-value = 0.0050


## Mini simulation: a toy power curve

This is a scaled-down version of what `simulations/ba_sweep.py` does --
regenerate two populations at varying separation (`gamma2`) and track how
often each test rejects. Here `B_REPLICATES` and `gamma2_values` are both
tiny purely so this cell finishes in seconds; the real thesis figures use
`B_REPLICATES=100` and 21 `gamma2` values (see `simulations/ba_sweep.py`).

In [9]:
gamma2_values = [2.5, 2.2, 2.0]  # 2.5 = null (equal to gamma1); further = larger effect
B_REPLICATES = 5                 # tiny, just to demonstrate the loop -- not a real power estimate
ALPHA = 0.05

rejection_rates = {"Ginestet": [], "Dubey": [], "Lovato": []}

for gamma2 in gamma2_values:
    pvals = {"Ginestet": [], "Dubey": [], "Lovato": []}
    for rep in range(B_REPLICATES):
        d = generate_data(n_samples=25, n_nodes=10, random_state=rep,
                           gamma1=2.5, gamma2=gamma2, m=2)
        G, labels = d["G_all"], d["y"]
        laps = [nx.laplacian_matrix(g).toarray() for g in G]

        pvals["Ginestet"].append(ginestet2017.run_test(G, labels, laplacians=laps)["p_value"])
        pvals["Dubey"].append(dubey2019.run_test(G, labels, laplacians=laps)["p_value"])
        pvals["Lovato"].append(lovato2020.run_test(G, labels, B=100, random_state=rep,
                                                     laplacians=laps)["p_combined"])

    for method in rejection_rates:
        rejection_rates[method].append(np.mean(np.array(pvals[method]) < ALPHA))

print(f"{'gamma2':>8}" + "".join(f"{m:>12}" for m in rejection_rates))
for i, g2 in enumerate(gamma2_values):
    row = "".join(f"{rejection_rates[m][i]:>12.2f}" for m in rejection_rates)
    print(f"{g2:>8}" + row)


  gamma2    Ginestet       Dubey      Lovato
     2.5        0.00        0.20        0.00
     2.2        0.60        0.20        0.20
     2.0        1.00        1.00        1.00


With only 5 replicates this is far too noisy to be a real power estimate
-- it's here purely to show the shape of the loop that produces the actual
power curves in the thesis. See `simulations/ba_sweep.py` for the full
version (`B_REPLICATES=100`, 21 `gamma2` values, parallelized across
workers, checkpointed to disk).